# RAG Assignment: Baseline vs. Re-Ranking

**Document:** Warhammer 40,000 Core Rules (cleaned into 294 chunks by `prepare_warhammer.py`, see `README.md`).

**Pipeline 1 (Baseline):** bi-encoder embeddings → top 10 by cosine similarity → top 3 to the LLM → **Answer 1**.

**Pipeline 2 (Re-Ranking):** same top 10 → cross-encoder scores → re-sorted top 3 to the LLM → **Answer 2**. *(next section)*

## 0. Setup

Install once: `pip install -r requirements.txt`

The OpenRouter key is read from the `OPENROUTER_API_KEY` environment variable so it never ends up in the notebook or the GitHub repo. Set it in PowerShell before starting Jupyter:

```powershell
$env:OPENROUTER_API_KEY = "sk-or-..."
jupyter notebook
```

In [1]:
import os

# Hugging Face download timeouts: if a download stalls (e.g. rate-limited), fail instead of hanging forever.
# Must be set BEFORE sentence_transformers / huggingface_hub are imported.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "15"   # seconds without receiving data before a download is abandoned
os.environ["HF_HUB_ETAG_TIMEOUT"] = "5"       # seconds to wait for HF to answer "is there a newer version?"
os.environ["HF_HUB_DISABLE_XET"] = "1"   # use plain HTTP downloads so the timeouts above apply

In [2]:
import json
import time
from pathlib import Path

import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = Path("output/warhammer_chunks.jsonl")
ANSWERS_PATH = Path("output/answers.json")   # cache of LLM answers (free tier is rate-limited)

BI_ENCODER_NAME = "BAAI/bge-small-en-v1.5" #stronger 
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

#BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2" #weaker
#QUERY_PREFIX = ""  

LLM_MODEL = "nvidia/nemotron-3.5-super-vl:free"   # free tier on OpenRouter; ":free" means no credits are charged

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT = 3     # chunks passed to the LLM

## 1. Load the chunked corpus

Each chunk's `text` field already starts with its heading (e.g. `SHOOTING PHASE › BIG GUNS NEVER TIRE`), so the heading is embedded together with the rule. Both pipelines use exactly this list.

In [3]:
with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

texts = [c["text"] for c in chunks]
print(f"{len(chunks)} chunks")
print({t: sum(c['type'] == t for c in chunks) for t in ('rule', 'summary', 'example')})
print()
print(chunks[91]["chunk_id"], "|", chunks[91]["text"][:200], "...")

294 chunks
{'rule': 217, 'summary': 66, 'example': 11}

wh-091 | SHOOTING PHASE › BIG GUNS NEVER TIRE
Monster and Vehicle units are eligible to shoot in their controlling player’s Shooting phase even while they are within Engagement Range of one or more enemy units ...


## 2. Pipeline 1, step a: index the chunks with a bi-encoder

The bi-encoder turns every chunk into one vector, **independently of any query**. This is done once, offline.

Vectors are L2-normalised, so the inner product of two vectors equals their cosine similarity. With 294 chunks a NumPy matrix is enough as the vector store (an exact search over all vectors); FAISS or Chroma would do the same thing at larger scale.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

# how many chunks would be cut off by the model's token limit?
n_tokens = [len(bi_encoder.tokenizer(t)["input_ids"]) for t in texts]
print(f"max_seq_length = {bi_encoder.max_seq_length}, longest chunk = {max(n_tokens)} tokens, "
      f"truncated chunks = {sum(n > bi_encoder.max_seq_length for n in n_tokens)}")

t0 = time.perf_counter()
chunk_vectors = bi_encoder.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
index_time = time.perf_counter() - t0

print(f"Index: {chunk_vectors.shape[0]} vectors x {chunk_vectors.shape[1]} dims, built in {index_time:.1f}s (one-off cost)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

max_seq_length = 512, longest chunk = 372 tokens, truncated chunks = 0


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Index: 294 vectors x 384 dims, built in 15.9s (one-off cost)


## 3. Pipeline 1, step b: retrieve the top 10

At query time only the query is encoded; its vector is compared to all chunk vectors with one matrix product. Encoding and search are timed separately for the latency table.

In [5]:
def retrieve(query, k=TOP_K_RETRIEVE):
    """Return the top-k chunks for a query as a list of dicts (rank, chunk_id, score, chunk), plus timings."""
    t0 = time.perf_counter()
    q_vec = bi_encoder.encode(QUERY_PREFIX + query, normalize_embeddings=True)
    t1 = time.perf_counter()
    scores = chunk_vectors @ q_vec              # cosine similarity with every chunk
    top = np.argsort(-scores)[:k]
    t2 = time.perf_counter()

    results = [{"rank": r + 1, "chunk_id": chunks[i]["chunk_id"], "score": float(scores[i]), "chunk": chunks[i]}
               for r, i in enumerate(top)]
    timings = {"query_encode_s": t1 - t0, "search_s": t2 - t1}
    return results, timings


def show_ranking(results, score_name="cosine"):
    for r in results:
        c = r["chunk"]
        print(f"#{r['rank']:<2} {r['chunk_id']}  {score_name}={r['score']:.3f}  [{c['type']}]  {c['heading']}")


retrieve("warm-up")   # first call is slower; keep it out of the timings

([{'rank': 1,
   'chunk_id': 'wh-208',
   'score': 0.586516261100769,
   'chunk': {'section': 'Stratagems',
    'heading': 'STRATAGEMS › RAPID INGRESS',
    'page_start': 42,
    'page_end': 42,
    'flavour': 'Be it cunning strategy, potent technology or supernatural ritual, there are many means by which a commander may hasten their warriors’ onset.',
    'type': 'rule',
    'part': 1,
    'text': 'STRATAGEMS › RAPID INGRESS\nWHEN: End of your opponent’s Movement phase.\nTARGET: One unit from your army that is in Reserves.\nEFFECT: Your unit can arrive on the battlefield as if it were the Reinforcements step of your Movement phase.\nRESTRICTIONS: You cannot use this Stratagem to enable a unit to arrive on the battlefield during a battle round it would not normally be able to do so in.',
    'chunk_id': 'wh-208',
    'n_words': 69}},
  {'rank': 2,
   'chunk_id': 'wh-122',
   'score': 0.5682833194732666,
   'chunk': {'section': 'Weapon Abilities',
    'heading': 'WEAPON ABILITIES › RAPI

## 4. Pipeline 1, step c: generate Answer 1 from the top 3

The LLM is reached through OpenRouter, which accepts requests from the `openai` client library. `generate_answer` holds the **one prompt template used by both pipelines**; only the chunks passed in change. Answers are cached to `output/answers.json` so re-running cells doesn't use up the free-tier rate limit.

In [6]:
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()   # copies the variables in .env into os.environ

# Read the key from .env; if it isn't there, ask for it (hidden input, never saved in the notebook)
api_key = os.environ.get("OPENROUTER_API_KEY") or getpass("OpenRouter API key: ")
llm = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

PROMPT_TEMPLATE = """You are a Warhammer 40,000 rules assistant.
Answer the question using ONLY the rules excerpts below. Quote the relevant rule and cite the chunk id in brackets.
If the excerpts do not contain the answer, say "The provided rules do not cover this."

RULES EXCERPTS:
{context}

QUESTION: {question}
ANSWER:"""

answer_cache = json.loads(ANSWERS_PATH.read_text(encoding="utf-8")) if ANSWERS_PATH.exists() else {}


def generate_answer(question, context_chunks, use_cache=True):
    """Ask the LLM, using only the given chunks as context. Returns (answer, seconds)."""
    context = "\n\n".join(f"[{c['chunk_id']}] {c['text']}" for c in context_chunks)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    key = f"{LLM_MODEL}|{question}|{','.join(c['chunk_id'] for c in context_chunks)}"
    if use_cache and key in answer_cache:
        return answer_cache[key]["answer"], answer_cache[key]["seconds"]

    t0 = time.perf_counter()
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,   # deterministic as far as possible, so Answer 1 vs 2 differ only because of the context
    )
    seconds = time.perf_counter() - t0
    answer = resp.choices[0].message.content

    answer_cache[key] = {"answer": answer, "seconds": seconds}
    ANSWERS_PATH.write_text(json.dumps(answer_cache, indent=2, ensure_ascii=False), encoding="utf-8")
    return answer, seconds

In [7]:
def pipeline1(question, verbose=True):
    """Baseline: bi-encoder top 10 -> first 3 straight to the LLM."""
    top10, timings = retrieve(question)
    top3 = [r["chunk"] for r in top10[:TOP_K_CONTEXT]]
    answer, llm_s = generate_answer(question, top3)
    timings["llm_s"] = llm_s

    if verbose:
        print(f"Q: {question}\n\nBi-encoder top {TOP_K_RETRIEVE}:")
        show_ranking(top10)
        print(f"\nAnswer 1:\n{answer}")
        print(f"\nretrieval {1000 * (timings['query_encode_s'] + timings['search_s']):.1f} ms, LLM {llm_s:.1f} s")
    return {"question": question, "top10": top10, "answer": answer, "timings": timings}

### Sanity check

A straightforward question first, to confirm the pipeline works end to end before trying failure queries.

In [8]:
p1 = pipeline1("What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?")

Q: What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?

Bi-encoder top 10:
#1  wh-098  cosine=0.879  [rule]  2. WOUND ROLL › WOUND ROLL ATTACK’S STRENGTH VS TARGET’S TOUGHNESS
#2  wh-097  cosine=0.860  [rule]  2. WOUND ROLL
#3  wh-095  cosine=0.752  [rule]  1. HIT ROLL
#4  wh-114  cosine=0.751  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#5  wh-096  cosine=0.744  [summary]  1. HIT ROLL
#6  wh-099  cosine=0.734  [rule]  2. WOUND ROLL › D6 RESULT REQUIRED
#7  wh-115  cosine=0.727  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#8  wh-107  cosine=0.722  [rule]  4. SAVING THROW › MORTAL WOUNDS
#9  wh-116  cosine=0.720  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#10 wh-117  cosine=0.709  [example]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING

Answer 1:
Strength is EQUAL to the Toughness: wound on 4+. [wh-098]

retrieval 32.5 ms, LLM 4.4 s


In [9]:
q = "What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?"
ans, _ = generate_answer(q, [], use_cache=False)
print(ans)   # want: "The provided rules do not cover this."

The provided rules do not cover this.


In [10]:
import copy

fake = copy.deepcopy(chunks[98])
fake["chunk_id"] = "wh-098-FAKE"   # different id, so the cache doesn't overwrite the real answer
fake["text"] = fake["text"].replace("EQUAL to the Toughness: wound on 4+", "EQUAL to the Toughness: wound on 5+")
assert fake["text"] != chunks[98]["text"], "replace didn't match"

ans, _ = generate_answer(q, [fake, chunks[97], chunks[95]], use_cache=False)
print(ans)   # want: 5+

Strength is EQUAL to the Toughness: wound on 5+. [wh-098-FAKE]


In [11]:
import re

def check_citations(answer, context_chunks):
    cited = set(re.findall(r"wh-\d{3}", answer))
    given = {c["chunk_id"] for c in context_chunks}
    print("cited:", sorted(cited), "| not in context:", sorted(cited - given) or "none")

check_citations(p1["answer"], [r["chunk"] for r in p1["top10"][:3]])

cited: ['wh-098'] | not in context: none


## 5. Pipeline 2: re-rank the top 10 with a cross-encoder

The cross-encoder reads the query and one chunk **together** in a single forward pass, so every query token can attend to every chunk token. That is more accurate than comparing two independently computed vectors, but it has to run once per (query, chunk) pair at query time, so we only apply it to the bi-encoder's 10 candidates.

The 10 candidates are the same ones Pipeline 1 retrieved; only their order changes. The new top 3 goes to the LLM with the identical prompt.

In [12]:
from sentence_transformers import CrossEncoder

# Try the strongest re-ranker first; if its download times out or is rate-limited, fall back to the next.
# The last one is already in the local cache from earlier runs, so it always loads.
CROSS_ENCODER_CANDIDATES = [
    "BAAI/bge-reranker-base",                  # ~1.1 GB, strongest
    "cross-encoder/ms-marco-MiniLM-L-12-v2",   # ~130 MB
    "cross-encoder/ms-marco-MiniLM-L-6-v2",    # model suggested in the brief (cached)
]

for name in CROSS_ENCODER_CANDIDATES:
    try:
        cross_encoder = CrossEncoder(name, max_length=512)
        CROSS_ENCODER_NAME = name
        break
    except Exception as e:
        print(f"Could not load {name}: {type(e).__name__}: {str(e)[:150]}")
else:
    raise RuntimeError("No cross-encoder could be loaded")

print(f"Using cross-encoder: {CROSS_ENCODER_NAME}")


def rerank(question, candidates):
    """Score each (question, chunk) pair with the cross-encoder and re-sort, best first.
    Keeps the bi-encoder rank as old_rank so rank shifts can be shown. Returns (reranked, seconds)."""
    t0 = time.perf_counter()
    pairs = [(question, r["chunk"]["text"]) for r in candidates]   # no bge prefix: that's only for the bi-encoder
    scores = cross_encoder.predict(pairs)                          # one relevance score (a logit) per pair
    seconds = time.perf_counter() - t0

    order = np.argsort(-scores)
    reranked = [{**candidates[i], "old_rank": candidates[i]["rank"], "rank": new + 1, "score": float(scores[i])}
                for new, i in enumerate(order)]
    return reranked, seconds


def show_rank_shift(reranked):
    """Print the cross-encoder order with each chunk's old bi-encoder rank."""
    for r in reranked:
        c = r["chunk"]
        move = r["old_rank"] - r["rank"]
        arrow = f"+{move}" if move > 0 else (str(move) if move < 0 else "=")
        entered = "  <-- moved into top 3" if r["rank"] <= TOP_K_CONTEXT < r["old_rank"] else ""
        print(f"#{r['old_rank']:<2} -> #{r['rank']:<2} ({arrow:>3})  {r['chunk_id']}  ce={r['score']:6.2f}  "
              f"[{c['type']}]  {c['heading']}{entered}")


rerank("warm-up", retrieve("warm-up")[0])   # first call is slower; keep it out of the timings

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

c:\course\credit\LMA\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\device\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Using cross-encoder: BAAI/bge-reranker-base


([{'rank': 1,
   'chunk_id': 'wh-067',
   'score': 3.887883576680906e-05,
   'chunk': {'section': 'Movement Phase',
    'heading': '2. REINFORCEMENTS',
    'page_start': 16,
    'page_end': 16,
    'flavour': '',
    'type': 'rule',
    'part': 1,
    'text': '2. REINFORCEMENTS\nSome units have a rule that allows them to start the battle in Reserves, such as Deep Strike (pg 39). Such units will arrive later in the battle and are known as Reserves units. Any Reserves units that have not been set up on the battlefield when the battle ends count as having been destroyed.\nIn the Reinforcements step of your Movement phase, if you have any Reserves units in your army, you can select one or more of them and set them up on the battlefield, one at a time. Once all your Reserves units that you wish to set up this turn have been set up, your Movement phase ends and you progress to your Shooting phase.\nDetails of how to set up Reserves units are described in the same rules that enabled the unit 

In [13]:
def compare(question):
    """Run Pipeline 1 and Pipeline 2 on the SAME bi-encoder top 10 and print both rankings and answers."""
    # shared first stage
    top10, t = retrieve(question)
    retrieval_s = t["query_encode_s"] + t["search_s"]

    # Pipeline 1: take the bi-encoder's top 3 as they are
    top3_p1 = [r["chunk"] for r in top10[:TOP_K_CONTEXT]]
    answer1, llm1_s = generate_answer(question, top3_p1)

    # Pipeline 2: re-sort the same 10 with the cross-encoder, take its top 3
    reranked, rerank_s = rerank(question, top10)
    top3_p2 = [r["chunk"] for r in reranked[:TOP_K_CONTEXT]]
    answer2, llm2_s = generate_answer(question, top3_p2)

    print(f"Q: {question}\n")
    print("Bi-encoder top 10 (Pipeline 1):")
    show_ranking(top10)
    print("\nCross-encoder re-ranking (Pipeline 2):  old rank -> new rank")
    show_rank_shift(reranked)
    print(f"\n--- Answer 1 (bi-encoder top 3: {[c['chunk_id'] for c in top3_p1]}) ---\n{answer1}")
    print(f"\n--- Answer 2 (re-ranked top 3: {[c['chunk_id'] for c in top3_p2]}) ---\n{answer2}")
    print(f"\nPipeline 1 retrieval: {1000 * retrieval_s:.1f} ms | "
          f"Pipeline 2 retrieval + re-rank: {1000 * (retrieval_s + rerank_s):.1f} ms "
          f"(re-rank alone {1000 * rerank_s:.1f} ms) | LLM: {llm1_s:.1f} s / {llm2_s:.1f} s")

    return {"question": question, "top10": top10, "reranked": reranked,
            "answer1": answer1, "answer2": answer2,
            "timings": {"retrieval_s": retrieval_s, "rerank_s": rerank_s, "llm1_s": llm1_s, "llm2_s": llm2_s}}

In [19]:
r_strength = compare("What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?")

Q: What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?

Bi-encoder top 10 (Pipeline 1):
#1  wh-098  cosine=0.879  [rule]  2. WOUND ROLL › WOUND ROLL ATTACK’S STRENGTH VS TARGET’S TOUGHNESS
#2  wh-097  cosine=0.860  [rule]  2. WOUND ROLL
#3  wh-095  cosine=0.752  [rule]  1. HIT ROLL
#4  wh-114  cosine=0.751  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#5  wh-096  cosine=0.744  [summary]  1. HIT ROLL
#6  wh-099  cosine=0.734  [rule]  2. WOUND ROLL › D6 RESULT REQUIRED
#7  wh-115  cosine=0.727  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#8  wh-107  cosine=0.722  [rule]  4. SAVING THROW › MORTAL WOUNDS
#9  wh-116  cosine=0.720  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#10 wh-117  cosine=0.709  [example]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING

Cross-encoder re-ranking (Pipeline 2):  old rank -> new rank
#1  -> #1  (  =)  wh-098  ce=  1.00  [rule]  2. WOUND ROLL › WOUND ROLL ATTACK’S

In [24]:
# Labelled questions for finding failure queries.
# Each entry: (question, {chunk ids that actually answer it}).
# Gold ids checked against output/warhammer_clean.md. Summary chunks (■ boxes) that restate the
# same rule count as correct too.
#
# A FAILURE QUERY is one where the gold chunk is at bi-encoder rank 4-10 (so Pipeline 1 misses it)
# and the cross-encoder moves it into the top 3 (so Pipeline 2 includes it).
# The comments say which chunk SHOULD win and which chunk is likely to beat it under the bi-encoder.

eval_set = [
    # ---------------- first round (kept so results stay comparable) ----------------
    # sanity check: question uses the rule's own words; both models should rank it #1
    ("What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?", {"wh-098"}),
    # exception: Big Guns Never Tire overrides the Locked in Combat ban (wh-088)
    ("Can my units shoot at an enemy Monster that is already fighting my other units in Engagement Range?", {"wh-091", "wh-093"}),
    # exception: Pistol overrides the Locked in Combat ban; many 'Engagement Range' chunks compete
    ("My unit is in Engagement Range of an enemy. Is there any weapon it can still shoot with?", {"wh-125", "wh-126"}),
    # exception: Assault overrides 'cannot shoot after Advancing' in wh-056
    ("My unit Advanced this turn. Can it still shoot?", {"wh-119", "wh-120"}),
    # nested exception: Characters/Monsters/Vehicles take 3 mortal wounds instead of losing a model
    ("What happens to a Character that rolls a 1 when firing a Hazardous weapon?", {"wh-140", "wh-141"}),
    # negation: 'cannot shoot ... in the same turn that it Fell Back'
    ("Is a unit prohibited from shooting in the turn it Fell Back?", {"wh-058"}),
    # negation: 'targets of a charge do NOT need to be visible'
    ("Do the targets of a charge need to be visible to the charging unit?", {"wh-152"}),
    # synonym: 'sprint' = Advance (first round: missed the top 10 entirely)
    ("How far can my troops run if they sprint during movement?", {"wh-056", "wh-057"}),
    # synonym: 'tank explodes' = Deadly Demise (first round: missed the top 10 entirely)
    ("Does a tank explode and hurt nearby units when it is destroyed?", {"wh-113"}),
    # synonym: 'not touching or near' = not in Engagement Range (first round: missed the top 10 entirely)
    ("Can a unit attack an enemy in close combat if it is not touching or near it?", {"wh-172"}),

    # ---------------- second round: aimed at the rank 4-10 zone ----------------
    # Exceptions & corner cases: phrased like the general rule, so the general rule ranks high
    # and the exception that actually answers the question is pushed down.

    # Pistol reworded: describes the situation instead of naming the weapon.
    # Likely distractors: Locked in Combat (wh-088), Engagement Range (wh-014)
    ("My infantry are stuck in Engagement Range with the enemy. What kind of weapon lets them still make a ranged attack?", {"wh-125", "wh-126"}),
    # Indirect Fire is the exception to needing visibility.
    # Likely distractors: the DETERMINING VISIBILITY chunks (wh-019 to wh-024)
    ("Can my unit shoot at an enemy unit it has no line of sight to?", {"wh-130", "wh-131"}),
    # Destroyed Transport passengers are NOT hit by Deadly Demise; they roll for mortal wounds instead.
    # Likely distractor: DEADLY DEMISE (wh-113), which shares 'destroyed' and 'explosion' wording
    ("If my Transport is destroyed, do the units inside get hit by its explosion?", {"wh-077", "wh-078"}),
    # Feel No Pain also works against mortal wounds.
    # Likely distractor: MORTAL WOUNDS (wh-107, wh-108)
    ("Can Feel No Pain stop mortal wounds?", {"wh-111", "wh-112"}),
    # Barricades let units fight from 2" across the terrain, an exception to normal Engagement Range.
    # Likely distractors: ENGAGEMENT RANGE (wh-014), fight-phase targeting (wh-172)
    ("Can my unit fight an enemy on the other side of a barricade that is more than 1 inch away?", {"wh-226"}),

    # Negation & constraints: the answer is 'no' or 'only if'; embeddings tend to blur this.

    # 'cannot embark if it has already disembarked ... in the same phase'
    # Likely distractor: DISEMBARK (wh-075, wh-076)
    ("Can my squad get back into its Transport in the same phase it got out?", {"wh-073", "wh-074"}),
    # invulnerable saves are NEVER modified by AP.
    # Likely distractor: SAVING THROW (wh-103), which explains how AP modifies armour saves
    ("Does Armour Penetration make an invulnerable save worse?", {"wh-105", "wh-106"}),
    # Heavy gives +1 to hit ONLY if the unit Remained Stationary.
    # Likely distractors: REMAIN STATIONARY (wh-053), HIT ROLL (wh-095)
    ("Do heavy weapons still get +1 to hit if my unit moved this turn?", {"wh-138", "wh-139"}),
    # Flying models can move over enemies but CANNOT end on top of them.
    # Likely distractors: NORMAL MOVES (wh-054), MOVING OVER TERRAIN (wh-062)
    ("Can a model that flies end its move on top of enemy models?", {"wh-064", "wh-065"}),
    # Deep Strike: must arrive MORE than 9" from all enemy models.
    # Likely distractors: RAPID INGRESS (wh-208), STRATEGIC RESERVES chunks
    ("Can my Deep Strike unit arrive closer than 9 inches to the enemy?", {"wh-191", "wh-192"}),

    # Keyword trap / synonyms: shares words with a wrong chunk, or uses player slang
    # (but keeps a few anchoring words, so the gold should still reach the top 10).

    # Precision lets attacks be allocated to a Character in an Attached unit.
    # Likely distractors: LEADER (wh-196, wh-197), ATTACHED UNITS (wh-048)
    ("Can I snipe the Character leading an enemy unit instead of hitting the bodyguards?", {"wh-132", "wh-133"}),
    # Fire Overwatch is the stratagem that lets you shoot in the opponent's turn.
    # Likely distractors: the SHOOTING PHASE chunks (wh-082 to wh-087)
    ("Can I shoot during my opponent's turn when their unit moves or charges?", {"wh-209"}),
    # Plunging Fire: shooting from 6"+ up in ruins improves AP.
    # Likely distractor: RUINS > BENEFIT OF COVER (wh-240) and the other cover chunks
    ("Do my models on the upper floors of a ruin get a bonus when shooting enemies on the ground?", {"wh-237"}),
    # Charge bonus = Fights First until end of turn.
    # Likely distractors: CHARGING WITH A UNIT (wh-152, wh-153), FIGHTS FIRST (wh-162) without the 'why'
    ("Does my unit get any advantage in the fight for having charged that turn?", {"wh-151"}),

    # Keyword trap: Melta and Rapid Fire both trigger at "half range" — Melta adds DAMAGE, Rapid Fire adds ATTACKS
    ("Which weapon ability increases the damage of an attack when the target is within half range?", {"wh-136", "wh-137"}),
    # Mirror of the above: the answer is Rapid Fire, Melta is the distractor
    ("Which weapon ability gives extra attacks when shooting at a target within half range?", {"wh-121", "wh-122"}),
    # Keyword trap: Lance (+1 to Wound after charging) vs Charge Bonus / Fights First (wh-151, wh-162)
    ("Do any weapons get a bonus to their wound roll if the unit charged this turn?", {"wh-129"}),
    # Keyword trap: Stealth (-1 to be hit) vs Big Guns Never Tire / Indirect Fire, which also subtract 1 from Hit rolls
    ("What ability makes enemy ranged attacks against my unit less likely to hit?", {"wh-090"}),
    # Exception reworded without 'Monster' (Big Guns Never Tire vs Locked in Combat wh-088)
    ("Can my tank shoot at enemy infantry that are in base contact with it?", {"wh-091", "wh-093"}),
    # Precision reworded closer to the rule's wording (was #5 -> #6)
    ("My attack wounded an enemy unit led by a Character. Can I allocate that wound to the Character?", {"wh-132", "wh-133"}),
    # Indirect Fire reworded with the rulebook's word 'visible' instead of 'line of sight' (was not in top 10)
    ("Can I attack an enemy unit that is not visible to any of my models?", {"wh-130", "wh-131"}),
]

print(f"{len(eval_set)} questions")

31 questions


In [31]:
rows = []
for question, gold in eval_set:
    top10, _ = retrieve(question)
    reranked, _ = rerank(question, top10)

    def gold_rank(results):
        ranks = [r["rank"] for r in results if r["chunk_id"] in gold]
        return min(ranks) if ranks else None

    bi, ce = gold_rank(top10), gold_rank(reranked)
    flag = "  <-- failure query candidate" if bi and ce and bi > TOP_K_CONTEXT >= ce else ""
    print(f"bi-encoder #{bi}  ->  cross-encoder #{ce}   {question}{flag}")
    rows.append({"question": question, "gold": sorted(gold), "bi_rank": bi, "ce_rank": ce})

# keep one results file per bi-encoder so the runs can be compared in the report
out = Path(f"output/eval_{BI_ENCODER_NAME.split('/')[-1]}__{CROSS_ENCODER_NAME.split('/')[-1]}.json")
out.write_text(json.dumps(rows, indent=2), encoding="utf-8")
print(f"\nsaved {out}")

bi-encoder #1  ->  cross-encoder #1   What do I need to roll to wound if my attack's Strength is equal to the target's Toughness?
bi-encoder #2  ->  cross-encoder #2   Can my units shoot at an enemy Monster that is already fighting my other units in Engagement Range?
bi-encoder #5  ->  cross-encoder #2   My unit is in Engagement Range of an enemy. Is there any weapon it can still shoot with?  <-- failure query candidate
bi-encoder #3  ->  cross-encoder #1   My unit Advanced this turn. Can it still shoot?
bi-encoder #1  ->  cross-encoder #1   What happens to a Character that rolls a 1 when firing a Hazardous weapon?
bi-encoder #1  ->  cross-encoder #1   Is a unit prohibited from shooting in the turn it Fell Back?
bi-encoder #1  ->  cross-encoder #1   Do the targets of a charge need to be visible to the charging unit?
bi-encoder #None  ->  cross-encoder #None   How far can my troops run if they sprint during movement?
bi-encoder #None  ->  cross-encoder #None   Does a tank explode and hu

In [27]:
r_pistol = compare("My unit is in Engagement Range of an enemy. Is there any weapon it can still shoot with?")


Q: My unit is in Engagement Range of an enemy. Is there any weapon it can still shoot with?

Bi-encoder top 10 (Pipeline 1):
#1  wh-092  cosine=0.866  [rule]  SHOOTING PHASE › BIG GUNS NEVER TIRE
#2  wh-088  cosine=0.850  [rule]  SHOOTING PHASE › Locked in Combat
#3  wh-091  cosine=0.836  [rule]  SHOOTING PHASE › BIG GUNS NEVER TIRE
#4  wh-089  cosine=0.831  [summary]  SHOOTING PHASE › Locked in Combat
#5  wh-126  cosine=0.812  [summary]  WEAPON ABILITIES › PISTOL
#6  wh-125  cosine=0.795  [rule]  WEAPON ABILITIES › PISTOL
#7  wh-093  cosine=0.789  [summary]  SHOOTING PHASE › BIG GUNS NEVER TIRE
#8  wh-014  cosine=0.765  [rule]  ENGAGEMENT RANGE
#9  wh-036  cosine=0.754  [rule]  SHOOTING PHASE
#10 wh-015  cosine=0.753  [summary]  ENGAGEMENT RANGE

Cross-encoder re-ranking (Pipeline 2):  old rank -> new rank
#1  -> #1  (  =)  wh-092  ce=  1.00  [rule]  SHOOTING PHASE › BIG GUNS NEVER TIRE
#6  -> #2  ( +4)  wh-125  ce=  1.00  [rule]  WEAPON ABILITIES › PISTOL  <-- moved into top 3
#2  ->

In [28]:
r_lance = compare("Do any weapons get a bonus to their wound roll if the unit charged this turn?")


Q: Do any weapons get a bonus to their wound roll if the unit charged this turn?

Bi-encoder top 10 (Pipeline 1):
#1  wh-139  cosine=0.730  [summary]  WEAPON ABILITIES › HEAVY
#2  wh-151  cosine=0.719  [rule]  CHARGE PHASE › CHARGE BONUS
#3  wh-120  cosine=0.713  [summary]  WEAPON ABILITIES › ASSAULT
#4  wh-207  cosine=0.709  [rule]  STRATAGEMS › TANK SHOCK
#5  wh-201  cosine=0.708  [rule]  STRATAGEMS › COMMAND RE-ROLL
#6  wh-212  cosine=0.708  [rule]  STRATAGEMS › HEROIC INTERVENTION
#7  wh-129  cosine=0.705  [rule]  WEAPON ABILITIES › LANCE
#8  wh-119  cosine=0.699  [rule]  WEAPON ABILITIES › ASSAULT
#9  wh-099  cosine=0.693  [rule]  2. WOUND ROLL › D6 RESULT REQUIRED
#10 wh-107  cosine=0.691  [rule]  4. SAVING THROW › MORTAL WOUNDS

Cross-encoder re-ranking (Pipeline 2):  old rank -> new rank
#7  -> #1  ( +6)  wh-129  ce=  0.94  [rule]  WEAPON ABILITIES › LANCE  <-- moved into top 3
#4  -> #2  ( +2)  wh-207  ce=  0.25  [rule]  STRATAGEMS › TANK SHOCK  <-- moved into top 3
#6  -> #3 

In [29]:
r_precision = compare("My attack wounded an enemy unit led by a Character. Can I allocate that wound to the Character?")

Q: My attack wounded an enemy unit led by a Character. Can I allocate that wound to the Character?

Bi-encoder top 10 (Pipeline 1):
#1  wh-101  cosine=0.792  [rule]  3. ALLOCATE ATTACK
#2  wh-197  cosine=0.749  [rule]  DEPLOYMENT ABILITIES › LEADER
#3  wh-102  cosine=0.748  [summary]  3. ALLOCATE ATTACK
#4  wh-133  cosine=0.741  [summary]  WEAPON ABILITIES › PRECISION
#5  wh-107  cosine=0.734  [rule]  4. SAVING THROW › MORTAL WOUNDS
#6  wh-198  cosine=0.730  [summary]  DEPLOYMENT ABILITIES › LEADER
#7  wh-115  cosine=0.723  [rule]  5. INFLICT DAMAGE › Hints and Tips FAST DICE ROLLING
#8  wh-120  cosine=0.721  [summary]  WEAPON ABILITIES › ASSAULT
#9  wh-196  cosine=0.712  [rule]  DEPLOYMENT ABILITIES › LEADER
#10 wh-048  cosine=0.708  [rule]  2. BATTLE-SHOCK › ATTACHED UNITS

Cross-encoder re-ranking (Pipeline 2):  old rank -> new rank
#2  -> #1  ( +1)  wh-197  ce=  0.95  [rule]  DEPLOYMENT ABILITIES › LEADER
#4  -> #2  ( +2)  wh-133  ce=  0.85  [summary]  WEAPON ABILITIES › PRECISION 

In [30]:
import statistics

failure_queries = [
    "My unit is in Engagement Range of an enemy. Is there any weapon it can still shoot with?",
    "Do any weapons get a bonus to their wound roll if the unit charged this turn?",
    "My attack wounded an enemy unit led by a Character. Can I allocate that wound to the Character?",
]
N_RUNS = 5   # repeat each measurement; single runs on a CPU are noisy

bi_ms, rerank_ms = [], []
for q in failure_queries:
    for _ in range(N_RUNS):
        top10, t = retrieve(q)
        bi_ms.append(1000 * (t["query_encode_s"] + t["search_s"]))
        _, s = rerank(q, top10)
        rerank_ms.append(1000 * s)

p2_ms = [b + r for b, r in zip(bi_ms, rerank_ms)]
llm_s = [v["seconds"] for v in answer_cache.values() if "FAKE" not in str(v)]   # every real LLM call so far

def row(name, xs, unit):
    print(f"{name:45} median {statistics.median(xs):9.1f} {unit}   min {min(xs):9.1f}   max {max(xs):9.1f}")

print(f"Latency over {len(failure_queries)} queries x {N_RUNS} runs (CPU)\n")
row("Pipeline 1: bi-encoder retrieval", bi_ms, "ms")
row("Pipeline 2: cross-encoder re-rank (10 pairs)", rerank_ms, "ms")
row("Pipeline 2: retrieval + re-rank", p2_ms, "ms")
row("LLM answer (OpenRouter free tier)", llm_s, "s ")

# Scaling question: what would it cost to run the cross-encoder over the WHOLE corpus for one query?
t0 = time.perf_counter()
cross_encoder.predict([(failure_queries[0], c["text"]) for c in chunks], batch_size=32)
full_s = time.perf_counter() - t0
print(f"\nCross-encoder over all {len(chunks)} chunks for ONE query: {full_s:.1f} s "
      f"(vs {statistics.median(rerank_ms) / 1000:.1f} s for 10 candidates)")

Latency over 3 queries x 5 runs (CPU)

Pipeline 1: bi-encoder retrieval              median      34.2 ms   min      25.3   max      47.9
Pipeline 2: cross-encoder re-rank (10 pairs)  median    3820.7 ms   min    3191.2   max    4800.7
Pipeline 2: retrieval + re-rank               median    3867.1 ms   min    3227.4   max    4834.3
LLM answer (OpenRouter free tier)             median      18.6 s    min       3.4   max     109.5

Cross-encoder over all 294 chunks for ONE query: 62.0 s (vs 3.8 s for 10 candidates)
